# 8. `link errors remain current` (switch_module, error)

In [1]:
from common import *

shape, log = show(8)

log shape 132: switch_module | error | link errors remain current
13,538 logs (3.3%) from 16 nodes, 2005-05-19 15:57:24 to 2006-04-30 09:10:07
log 379250 at 2005-11-18 21:38:39: 228944 Interconnect-1N00 switch_module error 1132349919 1 link errors remain current


In [2]:
print("Uh oh.")

Uh oh.


What this switch logged in the 2 days before

In [3]:
display(timeline(log.node, log.ts - 2 * day, log.ts))

,first_ts,last_ts,count,component,state,template
0,2005-11-16 21:38:48,2005-11-18 14:41:18,227,switch_module,error,link errors remain current
1,2005-11-18 14:45:41,2005-11-18 14:45:41,1,switch_module,bcast-error,Link error on broadcast tree <NODE>:<N>:<N>:<N>
2,2005-11-18 14:50:46,2005-11-18 21:38:39,58,switch_module,error,link errors remain current


Per hour over the 7 days before: this line on this switch

In [4]:
month = (log.ts - 7 * day, log.ts)
per_hour(*month, node=log.node, log_shape_id=shape.log_shape_id)

2005-11-11 22    2.0
2005-11-11 23    2.0
2005-11-12 00    2.0
2005-11-12 01    2.0
2005-11-12 03    1.0
                ... 
2005-11-18 17    7.0
2005-11-18 18    9.0
2005-11-18 19    9.0
2005-11-18 20    8.0
2005-11-18 21    7.0
Length: 88, dtype: float64

Seems like it is constantly raising these logs.

I have no idea what this is. Let's see what kinds of messages these switch modules can even emit in general.
- The messages of all the switches over the whole history, by first appearance

In [5]:
switch_shapes = find_shapes(component="switch_module").sort_values("first_seen")
display(switch_shapes[["state", "template", "n_logs", "n_nodes", "first_seen", "last_seen"]])

,state,template,n_logs,n_nodes,first_seen,last_seen
4,error,Link in reset,2057,12,2003-12-26 12:30:20,2004-01-13 19:15:16
5,error,Link ok,1965,12,2003-12-26 12:30:21,2004-01-13 19:15:23
6,error,Link error,1481,12,2003-12-26 12:54:53,2004-01-13 17:50:11
9,control,power/control problem,355,16,2003-12-28 17:42:05,2005-03-26 14:36:30
11,control,power/control ok,79,16,2003-12-28 17:42:56,2005-03-26 14:49:44
0,error,Linkerror event interval expired,80895,16,2004-01-14 16:43:12,2005-05-18 16:40:59
3,bcast-error,Link error,2267,7,2004-01-14 16:43:22,2005-05-06 18:10:12
10,bcast-error,Link in reset,80,9,2004-01-16 02:28:56,2005-03-26 14:26:47
12,bcast-error,Link ok,69,3,2004-01-16 02:28:58,2005-03-26 14:47:49
15,psu,PSU status ( on off ),26,4,2004-06-22 16:47:37,2006-04-20 14:27:53


Looking at the first seen and last seen dates, it looks like the message texts for these components have changed over time, probably due to software/hardware updates.
#### Anyways, I don't see how this helps us. We might have a problem in this switch. let's see what's going on in the cluster:

In [6]:
shapes_between(log.ts - hour, log.ts)

,component,state,template,n_logs,n_sources,first,last
0,switch_module,bcast-error,Link error on broadcast tree <NODE>:<N>:<N>:<N>,8,1,2005-11-18 20:39:07,2005-11-18 21:38:19
1,switch_module,error,link errors remain current,21,6,2005-11-18 20:42:15,2005-11-18 21:38:39
2,gige,temperature,normal,3,3,2005-11-18 21:04:17,2005-11-18 21:35:16
3,gige,temperature,warning,1,1,2005-11-18 21:27:17,2005-11-18 21:27:17


In [7]:
logs.search(log.ts - hour, log.ts)

[Log(id=379250, event_id=228944, node='Interconnect-1N00', component='switch_module', state='error', time=1132349919.0, flag=1, message='link errors remain current', bad_time=False, ts=datetime.datetime(2005, 11, 18, 21, 38, 39), original_log='228944 Interconnect-1N00 switch_module error 1132349919 1 link errors remain current', log_shape_id=132, n_copies=1),
 Log(id=66595, event_id=228943, node='Interconnect-1T00', component='switch_module', state='bcast-error', time=1132349899.0, flag=1, message='Link error on broadcast tree Interconnect-1T00:00:2:5', bad_time=False, ts=datetime.datetime(2005, 11, 18, 21, 38, 19), original_log='228943 Interconnect-1T00 switch_module bcast-error 1132349899 1 Link error on broadcast tree Interconnect-1T00:00:2:5', log_shape_id=56, n_copies=1),
 Log(id=66594, event_id=228942, node='Interconnect-1T00', component='switch_module', state='bcast-error', time=1132349899.0, flag=1, message='Link error on broadcast tree Interconnect-1T00:00:1:5', bad_time=False

OK. I see this is happenning a lot on a few 'Interconnect-1N0_' switches, and then there is also a 'Interconnect-1T00' switch displaying a different error message.

### Searched the web and found:
- Interconnect: compute nodes have two networks. A plain Ethernet one for management, and a fast low-latency one that MPI jobs use to talk to each other. This cluster is an AlphaServer SC with a Quadrics interconnect, so this is the fast one.
- Switch module: one chassis of that interconnect. Interconnect-0N02 is the reporter, so the "16 nodes" for this shape are 16 switch chassis, not compute nodes.
- My reading of the name is that N is a node-level switch (compute nodes plug into it) and T is a top-level switch that joins the node-level ones.
- Link errors: every cable between a node and a switch (or switch and switch) checks data for corruption. A small trickle of errors is normal and gets retransmitted silently. A sustained stream means a bad cable, port or adapter, and ends in a link reset or a dead MPI job.

TODO - continue